# Document Embedding Analysis — step-by-step demo

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/document_embedding_analysis/blob/master/scripts/step_by_step_demo.ipynb)

This notebook demonstrates the main usages of `document_embedding_analysis` and the optional **DEA-aware qualitative LLM judge**.

It is intended to live at:

```text
scripts/step_by_step_demo.ipynb
```

## Validation checklist

- [ ] Install the repo in Colab or use the local checkout.
- [ ] Load one DEA target JSON from public `output/` examples, or use a synthetic fallback.
- [ ] Inspect target plan/content/bibliography.
- [ ] Create weak and stronger generated Markdown candidates.
- [ ] Run `evaluate_document(...)` quickly without full DEA embeddings.
- [ ] Optionally run full DEA target-distance scoring.
- [ ] Run the new DEA-aware judge once per evaluation.
- [ ] Compare candidates as scalar and multi-objective feedback.

## Secrets

Use Colab Secrets or environment variables:

- `OPENROUTER_API_KEY` — preferred for the judge.
- `OPENAI_API_KEY` — optional for OpenAI embeddings or OpenAI fallback.

If no key is available, a deterministic fake judge is used so the notebook remains runnable.


## 0. Runtime knobs

In [1]:
REPO_URL = "https://github.com/doxav/document_embedding_analysis.git"
REPO_BRANCH = "master"
OPENROUTER_MODEL_DEFAULT = "openai/gpt-5-nano"
RUN_FULL_DEA = False          # set True to load/use embeddings against the target JSON
RUN_LOW_LEVEL_COMPARISON = False
N_SECTIONS_FOR_CANDIDATE = 5


## 1. Install repository and dependencies

In [2]:
import os, sys, subprocess, pathlib, json, inspect, time, textwrap

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print("IN_COLAB =", IN_COLAB)

if IN_COLAB:
    repo_dir = pathlib.Path("/content/document_embedding_analysis")
    if not repo_dir.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)])
    os.chdir(repo_dir)
    install_commands = [
        [sys.executable, "-m", "pip", "install", "-q", "-e", "."],
        [sys.executable, "-m", "pip", "install", "-q", "openai", "requests", "rouge-score", "python-dotenv", "scipy", "scikit-learn", "loguru", "beautifulsoup4", "pandas"],
    ]
else:
    cwd = pathlib.Path.cwd()
    if cwd.name == "scripts" and (cwd.parent / "common").exists():
        os.chdir(cwd.parent)
    install_commands = []

print("Working directory:", pathlib.Path.cwd())

for cmd in install_commands:
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError as exc:
        print("Install command failed, continuing:", " ".join(cmd), exc)

if RUN_FULL_DEA:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers", "torch", "langchain-community", "langchain-openai"])

if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))


IN_COLAB = False
Working directory: /home/user/code/document_embedding_analysis


## 2. Load Colab secrets / environment variables

In [3]:
def load_secret_to_env(name: str) -> bool:
    if os.environ.get(name):
        return True
    if IN_COLAB:
        try:
            from google.colab import userdata  # type: ignore
            value = userdata.get(name)
            if value:
                os.environ[name] = value
                return True
        except Exception as exc:
            print(f"Could not read Colab secret {name}: {exc}")
    return bool(os.environ.get(name))

print("OPENROUTER_API_KEY available:", load_secret_to_env("OPENROUTER_API_KEY"))
print("OPENAI_API_KEY available:", load_secret_to_env("OPENAI_API_KEY"))
print("OPENROUTER_MODEL:", os.environ.get("OPENROUTER_MODEL", OPENROUTER_MODEL_DEFAULT))


OPENROUTER_API_KEY available: True
OPENAI_API_KEY available: True
OPENROUTER_MODEL: openai/gpt-5-nano


## 3. Import evaluator and detect judge support

In [4]:
from common.doc_eval import evaluate_document, temporary_transform_dea_into_markdown

sig = inspect.signature(evaluate_document)
SUPPORTS_DEA_JUDGE = "use_dea_judge" in sig.parameters
print("evaluate_document supports use_dea_judge:", SUPPORTS_DEA_JUDGE)
print(sig)

try:
    from common.dea_judge import run_dea_judge
    SUPPORTS_DIRECT_DEA_JUDGE = True
except Exception as exc:
    run_dea_judge = None
    SUPPORTS_DIRECT_DEA_JUDGE = False
    print("common.dea_judge not available:", exc)
print("Direct run_dea_judge available:", SUPPORTS_DIRECT_DEA_JUDGE)


evaluate_document supports use_dea_judge: True
(document_content: 'str', turns: 'List | None' = None, solution: 'dict | None' = None, content_type: 'str' = 'markdown', *, use_enhanced_metrics: 'bool' = False, skip_dea: 'bool' = False, openai_model: 'str | None' = None, prometheus_version: 'str' = 'v2.0', dea_embedding_backend: 'str | None' = None, dea_embedding_model: 'str | None' = None, golden_entities: 'List[str] | None' = None, skip_entity_recall: 'bool' = True, use_dea_judge: 'bool' = True, dea_judge_model: 'str | None' = None, dea_judge_client=None, dea_judge_lm=None, dea_judge_max_prompt_chars: 'int' = 20000)
Direct run_dea_judge available: True


## 4. Load one target DEA JSON from GitHub output examples

The notebook tries `output/wikipedia`, `output/latex`, `output/arxiv`, `output/patent`, then uses a synthetic fallback if no public JSON is available.


In [5]:
import requests
from pprint import pprint

GITHUB_REPO = "doxav/document_embedding_analysis"
GITHUB_REF = "master"
OUTPUT_DIRS = ["output/wikipedia", "output/latex", "output/arxiv", "output/patent", "output/freshwiki"]

def list_github_dir(path: str):
    url = f"https://api.github.com/repos/{GITHUB_REPO}/contents/{path}?ref={GITHUB_REF}"
    r = requests.get(url, timeout=30)
    if r.status_code != 200:
        return []
    data = r.json()
    return data if isinstance(data, list) else []

def load_first_public_target():
    for out_dir in OUTPUT_DIRS:
        items = list_github_dir(out_dir)
        json_items = [x for x in items if x.get("name", "").endswith(".json") and x.get("download_url")]
        if not json_items:
            continue
        json_items.sort(key=lambda x: x.get("size", 10**18))
        item = json_items[0]
        print(f"Loading target: {out_dir}/{item['name']} ({item.get('size', '?')} bytes)")
        rr = requests.get(item["download_url"], timeout=60)
        rr.raise_for_status()
        target = rr.json()
        target_path = pathlib.Path("/tmp") / item["name"]
        target_path.write_text(json.dumps(target), encoding="utf-8")
        target["target_file_path"] = str(target_path)
        return target, item
    raise FileNotFoundError("No target JSON found in public output folders")

def synthetic_target():
    target = {
        "id": "synthetic-demo",
        "title": "Solar Power Basics",
        "context": "Explain solar power systems, their operation, applications, and limitations.",
        "abstract": "Explain solar power systems, their operation, applications, and limitations.",
        "plan": [
            {"section_id": 1, "section": "Introduction", "content": "Solar power converts sunlight into usable electricity or heat. It is a key renewable energy technology used in homes, industry, and utility-scale generation.", "resources_used": [1]},
            {"section_id": 2, "section": "Photovoltaic effect", "content": "Photovoltaic cells generate electric current when photons excite electrons in semiconductor materials. Cell design, materials, and module wiring affect efficiency.", "resources_used": [1]},
            {"section_id": 3, "section": "System components", "content": "A solar power system usually includes panels, inverters, mounting, wiring, monitoring, and sometimes batteries or grid interconnection equipment.", "resources_used": [2]},
            {"section_id": 4, "section": "Applications", "content": "Applications include rooftop systems, solar farms, off-grid power, water pumping, satellites, and hybrid microgrids.", "resources_used": []},
            {"section_id": 5, "section": "Limitations", "content": "Solar generation varies with weather, time of day, shading, latitude, and storage availability. Integration requires planning for intermittency.", "resources_used": [2]},
        ],
        "resources": [
            {"resource_id": 1, "resource_description": "NREL overview of photovoltaic solar energy."},
            {"resource_id": 2, "resource_description": "IEA report on solar PV systems and grid integration."},
        ],
    }
    target_path = pathlib.Path("/tmp/synthetic_dea_target.json")
    target_path.write_text(json.dumps(target), encoding="utf-8")
    target["target_file_path"] = str(target_path)
    return target, {"name": "synthetic_dea_target.json", "download_url": None}

try:
    solution, source_item = load_first_public_target()
except Exception as exc:
    print("Using synthetic fallback:", exc)
    solution, source_item = synthetic_target()

print("Title:", solution.get("title"))
print("Sections:", len(solution.get("plan", [])))
print("Resources:", len(solution.get("resources", [])))


Loading target: output/wikipedia/Dual-phase evolution.json (3231335 bytes)
Title: Dual-phase evolution
Sections: 12
Resources: 10


## 5. Inspect target structure

In [6]:
print("Title:", solution.get("title"))
print("Context excerpt:", (solution.get("context") or solution.get("abstract") or "")[:400])
print("\nPlan preview:")
for i, sec in enumerate(solution.get("plan", [])[:8], 1):
    title = sec.get("section") or sec.get("title") or f"Section {i}"
    content = sec.get("content", "")
    print(f"{i}. {title} | chars={len(content)} | resources_used={sec.get('resources_used', [])}")
    print("   ", content[:180].replace("\n", " "), "...")
print("\nResources preview:")
for r in solution.get("resources", [])[:5]:
    compact_resource = {k: v for k, v in r.items() if "embedding" not in k}
    pprint(compact_resource)


Title: Dual-phase evolution
Context excerpt: Dual phase evolution (DPE) is a process that drives self-organization within complex adaptive systems.[1] It arises in response to phase changes within the network of connections formed by a system's components. DPE occurs in a wide range of physical, biological and social systems. Its applications to technology include methods for manufacturing novel materials and algorithms to solve complex prob

Plan preview:
1. h2 Introduction | chars=1418 | resources_used=[2]
    Dual phase evolution (DPE) is a process that promotes the emergence of large-scale order in complex systems. It occurs when a system repeatedly switches between various kinds of ph ...
2. h2 The DPE mechanism | chars=58 | resources_used=[1]
    The following features are necessary for DPE to occur.[1]  ...
3. h3 Underlying network | chars=1094 | resources_used=[]
    DPE occurs where a system has an underlying network. That is, the system's components form a set of nodes and th

## 6. Create weak and better candidate Markdown documents

In [7]:
def section_title(section, i):
    return section.get("section") or section.get("title") or f"Section {i}"

def make_candidate(solution, strength="weak", n_sections=5):
    title = solution.get("title") or "Candidate document"
    plan = solution.get("plan", [])[:n_sections]
    lines = [f"# {title}", ""]
    if strength == "weak":
        for i, sec in enumerate(plan[:max(1, min(2, len(plan)))], 1):
            lines += [f"## {section_title(sec, i)}", (sec.get("content", "")[:180] or "Short overview.").strip(), ""]
        lines += ["## Extra notes", "This generic section does not clearly map to the target plan."]
        return "\n".join(lines)
    for i, sec in enumerate(plan, 1):
        text = sec.get("content", "")[:900] or "Content unavailable."
        refs = sec.get("resources_used", []) or []
        if refs:
            text += " " + " ".join(f"[{r}]" for r in refs[:3])
        lines += [f"## {section_title(sec, i)}", text.strip(), ""]
    if solution.get("resources"):
        lines.append("## References")
        for r in solution.get("resources", [])[:8]:
            rid = r.get("resource_id") or r.get("id") or "?"
            desc = r.get("resource_description") or r.get("name") or r.get("title") or r.get("description") or str(r)[:120]
            lines.append(f"{rid}. {desc}")
    return "\n".join(lines)

weak_candidate = make_candidate(solution, "weak", N_SECTIONS_FOR_CANDIDATE)
better_candidate = make_candidate(solution, "better", N_SECTIONS_FOR_CANDIDATE)
print("WEAK CANDIDATE\n", weak_candidate[:1000])
print("\n" + "="*80 + "\nBETTER CANDIDATE\n", better_candidate[:1000])


WEAK CANDIDATE
 # Dual-phase evolution

## h2 Introduction
Dual phase evolution (DPE) is a process that promotes the emergence of large-scale order in complex systems. It occurs when a system repeatedly switches between various kinds of ph

## h2 The DPE mechanism
The following features are necessary for DPE to occur.[1]

## Extra notes
This generic section does not clearly map to the target plan.

BETTER CANDIDATE
 # Dual-phase evolution

## h2 Introduction
Dual phase evolution (DPE) is a process that promotes the emergence of large-scale order in complex systems. It occurs when a system repeatedly switches between various kinds of phases, and in each phase different processes act on the components or connections in the system. DPE arises because of a property of graphs and networks: the connectivity avalanche that occurs in graphs as the number of edges increases.[2]
 Social networks provide a familiar example. In a social network the nodes of the network are people and the network c

## 7. Fast evaluation with `evaluate_document(skip_dea=True)`

This runs without loading full embedding models. It still computes article-level metrics against a Markdown version of the gold solution.


In [8]:
def call_eval(candidate, *, use_dea_judge=False, judge_kwargs=None, skip_dea=True):
    kwargs = dict(
        document_content=candidate,
        solution=solution,
        content_type="markdown",
        skip_dea=skip_dea,
        use_enhanced_metrics=False,
        skip_entity_recall=True,
    )
    if SUPPORTS_DEA_JUDGE:
        kwargs["use_dea_judge"] = use_dea_judge
        kwargs.update(judge_kwargs or {})
    return evaluate_document(**kwargs)

weak_fast = call_eval(weak_candidate, skip_dea=True)
better_fast = call_eval(better_candidate, skip_dea=True)
print("Weak article metrics:")
pprint(weak_fast.get("article_metrics", {}))
print("Better article metrics:")
pprint(better_fast.get("article_metrics", {}))


INFO:common.doc_eval:Initializing evaluators...
INFO:common.doc_eval:Getting article quality metrics...
INFO:absl:Using default tokenizer.
INFO:common.doc_eval:Article metrics obtained: {'entity_recall': None, 'citation_count': 1, 'rouge_scores': {'rouge-1': {'p': 0.8387096774193549, 'r': 0.02333931777378815, 'f': 0.04541484716157205}, 'rouge-2': {'p': 0.7704918032786885, 'r': 0.021104625056129322, 'f': 0.04108391608391609}, 'rouge-l': {'p': 0.8387096774193549, 'r': 0.02333931777378815, 'f': 0.04541484716157205}}}
INFO:common.doc_eval:Initializing evaluators...
INFO:common.doc_eval:Getting article quality metrics...
INFO:absl:Using default tokenizer.



=== Document Evaluation Results ===

Prometheus Scores (0-1 scale, higher is better):

WriteHere Scores (0-1 scale, higher is better):

Article Quality Metrics:
Entity Recall (0-1 scale): N/A
Citation Count: 1

ROUGE Scores (precision, recall, F1):
rouge-1:
  p: 0.84
  r: 0.02
  f: 0.05
rouge-2:
  p: 0.77
  r: 0.02
  f: 0.04
rouge-l:
  p: 0.84
  r: 0.02
  f: 0.05


INFO:common.doc_eval:Article metrics obtained: {'entity_recall': None, 'citation_count': 8, 'rouge_scores': {'rouge-1': {'p': 0.9807692307692307, 'r': 0.3891382405745063, 'f': 0.5571979434447302}, 'rouge-2': {'p': 0.9580973952434881, 'r': 0.37988325101032777, 'f': 0.5440514469453376}, 'rouge-l': {'p': 0.9785067873303167, 'r': 0.3882405745062837, 'f': 0.5559125964010283}}}



=== Document Evaluation Results ===

Prometheus Scores (0-1 scale, higher is better):

WriteHere Scores (0-1 scale, higher is better):

Article Quality Metrics:
Entity Recall (0-1 scale): N/A
Citation Count: 8

ROUGE Scores (precision, recall, F1):
rouge-1:
  p: 0.98
  r: 0.39
  f: 0.56
rouge-2:
  p: 0.96
  r: 0.38
  f: 0.54
rouge-l:
  p: 0.98
  r: 0.39
  f: 0.56
Weak article metrics:
{'citation_count': 1,
 'entity_recall': None,
 'rouge_scores': {'rouge-1': {'f': 0.04541484716157205,
                              'p': 0.8387096774193549,
                              'r': 0.02333931777378815},
                  'rouge-2': {'f': 0.04108391608391609,
                              'p': 0.7704918032786885,
                              'r': 0.021104625056129322},
                  'rouge-l': {'f': 0.04541484716157205,
                              'p': 0.8387096774193549,
                              'r': 0.02333931777378815}}}
Better article metrics:
{'citation_count': 8,
 'entity_reca

## 8. Scalar and multi-objective scoring helpers

In [9]:
def nested(d, path, default=0.0):
    cur = d
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur

def scalar_score(scores):
    dea = scores.get("dea_evaluation_scores", {}) or {}
    article = scores.get("article_metrics", {}) or {}
    plan = dea.get("plan_embedding_similarity") or dea.get("global_plan_embedding_similarity") or 0.0
    content = dea.get("plan_contents_embedding_similarity") or dea.get("global_plan_contents_embedding_similarity") or 0.0
    refs = dea.get("plan_resources_embedding_similarity") or dea.get("resources_citation_coverage_score") or 0.0
    rouge_l = nested(article, ["rouge_scores", "rouge-l", "f"], 0.0)
    return 0.30*float(plan) + 0.45*float(content) + 0.15*float(refs) + 0.10*float(rouge_l)

def score_vector(scores):
    dea = scores.get("dea_evaluation_scores", {}) or {}
    article = scores.get("article_metrics", {}) or {}
    return {
        "scalar": scalar_score(scores),
        "plan": dea.get("plan_embedding_similarity") or dea.get("global_plan_embedding_similarity") or 0.0,
        "content": dea.get("plan_contents_embedding_similarity") or dea.get("global_plan_contents_embedding_similarity") or 0.0,
        "references": dea.get("plan_resources_embedding_similarity") or dea.get("resources_citation_coverage_score") or 0.0,
        "rouge_l_f": nested(article, ["rouge_scores", "rouge-l", "f"], 0.0),
        "citation_count": article.get("citation_count", 0),
    }

print("Weak vector:")
pprint(score_vector(weak_fast))
print("Better vector:")
pprint(score_vector(better_fast))


Weak vector:
{'citation_count': 1,
 'content': 0.0,
 'plan': 0.0,
 'references': 0.0,
 'rouge_l_f': 0.04541484716157205,
 'scalar': 0.004541484716157205}
Better vector:
{'citation_count': 8,
 'content': 0.0,
 'plan': 0.0,
 'references': 0.0,
 'rouge_l_f': 0.5559125964010283,
 'scalar': 0.055591259640102836}


## 9. Optional full DEA scoring

Set `RUN_FULL_DEA=True` at the top. This may download local embeddings or use OpenAI embeddings.


In [10]:
if RUN_FULL_DEA:
    backend = "openai" if os.environ.get("OPENAI_API_KEY") else "hf"
    print("Running full DEA with backend:", backend)
    t0 = time.perf_counter()
    weak_full = evaluate_document(
        document_content=weak_candidate,
        solution=solution,
        content_type="markdown",
        skip_dea=False,
        use_enhanced_metrics=False,
        dea_embedding_backend=backend,
        skip_entity_recall=True,
        **({"use_dea_judge": False} if SUPPORTS_DEA_JUDGE else {}),
    )
    better_full = evaluate_document(
        document_content=better_candidate,
        solution=solution,
        content_type="markdown",
        skip_dea=False,
        use_enhanced_metrics=False,
        dea_embedding_backend=backend,
        skip_entity_recall=True,
        **({"use_dea_judge": False} if SUPPORTS_DEA_JUDGE else {}),
    )
    print("Runtime:", time.perf_counter() - t0)
    print("Weak DEA scores:")
    pprint(weak_full.get("dea_evaluation_scores", {}))
    print("Better DEA scores:")
    pprint(better_full.get("dea_evaluation_scores", {}))
else:
    print("RUN_FULL_DEA is False; skipping full DEA target-distance scoring.")
    weak_full = better_full = None


RUN_FULL_DEA is False; skipping full DEA target-distance scoring.


## 10. Configure OpenRouter/OpenAI/fake judge

OpenRouter uses the OpenAI SDK pointed at `https://openrouter.ai/api/v1`.


In [11]:
from openai import OpenAI

def make_openrouter_client():
    if not os.environ.get("OPENROUTER_API_KEY"):
        return None, None
    model = os.environ.get("OPENROUTER_MODEL", OPENROUTER_MODEL_DEFAULT)
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
        default_headers={
            "HTTP-Referer": "https://github.com/doxav/document_embedding_analysis",
            "X-OpenRouter-Title": "document_embedding_analysis step-by-step demo",
        },
    )
    return client, model

def fake_judge_lm(messages, temperature=0):
    return json.dumps({
        "qualitative_assessment": "Fake judge: the candidate is relevant but may miss parts of the plan and citation support.",
        "keep": [{"point": "Topic preserved", "evidence": "The candidate keeps the title/topic.", "why_keep": "This anchors the document to the target."}],
        "problems": [
            {"issue": "The candidate appears structurally incomplete.", "main_impact": "plan", "priority": "P1 wrong", "impact": "Missing sections reduce plan/content coverage.", "confidence": "medium", "evidence": "The weak candidate contains fewer sections than the target preview."},
            {"issue": "Citation or bibliography support is weak.", "main_impact": "bibliography", "priority": "P1 wrong", "impact": "Claims may not be source-grounded.", "confidence": "medium", "evidence": "No or limited references are visible in the candidate."},
        ],
        "uncertainties": [{"question": "Are individual claims factually supported?", "needed_evidence": "Claim-level source passages or NLI/QA checks."}],
    })

openrouter_client, openrouter_model = make_openrouter_client()
if openrouter_client:
    judge_mode = "openrouter"
    judge_kwargs = {"dea_judge_client": openrouter_client, "dea_judge_model": openrouter_model}
elif os.environ.get("OPENAI_API_KEY"):
    judge_mode = "openai_model"
    judge_kwargs = {"dea_judge_model": os.environ.get("OPENAI_JUDGE_MODEL", "gpt-5-nano")}
else:
    judge_mode = "fake_lm"
    judge_kwargs = {"dea_judge_lm": fake_judge_lm, "dea_judge_model": "fake"}
print("Judge mode:", judge_mode)
print("Judge model:", judge_kwargs.get("dea_judge_model"))


Judge mode: openrouter
Judge model: openai/gpt-5-nano


## 11. Run the DEA-aware qualitative judge once per candidate

In [12]:
if SUPPORTS_DEA_JUDGE:
    weak_with_judge = call_eval(weak_candidate, use_dea_judge=True, judge_kwargs=judge_kwargs, skip_dea=True)
    better_with_judge = call_eval(better_candidate, use_dea_judge=True, judge_kwargs=judge_kwargs, skip_dea=True)
elif SUPPORTS_DIRECT_DEA_JUDGE:
    print("No evaluate_document hook; using direct run_dea_judge fallback for the weak candidate.")
    weak_with_judge = weak_fast
    weak_with_judge["dea_judge"] = run_dea_judge(
        document_content=weak_candidate,
        solution=solution,
        content_type="markdown",
        dea_scores=weak_fast.get("dea_evaluation_scores", {}),
        article_metrics=weak_fast.get("article_metrics", {}),
        prometheus_scores=weak_fast.get("prometheus_scores", {}),
        writehere_scores=weak_fast.get("writehere_scores", {}),
        lm=fake_judge_lm,
    )
    better_with_judge = better_fast
else:
    print("DEA judge feature is not available on this branch.")
    weak_with_judge = weak_fast
    better_with_judge = better_fast

print("Weak judge status:", weak_with_judge.get("dea_judge", {}).get("status"))
print("Better judge status:", better_with_judge.get("dea_judge", {}).get("status"))
if judge_mode == "openrouter":
    for candidate_name, result in (("weak", weak_with_judge), ("better", better_with_judge)):
        status = result.get("dea_judge", {}).get("status")
        if status != "ok":
            print(f"OpenRouter judge returned {status!r} for {candidate_name}; evaluation continued safely.")
            print("error:", result.get("dea_judge", {}).get("error"))


INFO:common.doc_eval:Initializing evaluators...
INFO:common.doc_eval:Getting article quality metrics...
INFO:absl:Using default tokenizer.
INFO:common.doc_eval:Article metrics obtained: {'entity_recall': None, 'citation_count': 1, 'rouge_scores': {'rouge-1': {'p': 0.8387096774193549, 'r': 0.02333931777378815, 'f': 0.04541484716157205}, 'rouge-2': {'p': 0.7704918032786885, 'r': 0.021104625056129322, 'f': 0.04108391608391609}, 'rouge-l': {'p': 0.8387096774193549, 'r': 0.02333931777378815, 'f': 0.04541484716157205}}}



=== Document Evaluation Results ===

Prometheus Scores (0-1 scale, higher is better):

WriteHere Scores (0-1 scale, higher is better):

Article Quality Metrics:
Entity Recall (0-1 scale): N/A
Citation Count: 1

ROUGE Scores (precision, recall, F1):
rouge-1:
  p: 0.84
  r: 0.02
  f: 0.05
rouge-2:
  p: 0.77
  r: 0.02
  f: 0.04
rouge-l:
  p: 0.84
  r: 0.02
  f: 0.05


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:common.doc_eval:Initializing evaluators...
INFO:common.doc_eval:Getting article quality metrics...
INFO:absl:Using default tokenizer.
INFO:common.doc_eval:Article metrics obtained: {'entity_recall': None, 'citation_count': 8, 'rouge_scores': {'rouge-1': {'p': 0.9807692307692307, 'r': 0.3891382405745063, 'f': 0.5571979434447302}, 'rouge-2': {'p': 0.9580973952434881, 'r': 0.37988325101032777, 'f': 0.5440514469453376}, 'rouge-l': {'p': 0.9785067873303167, 'r': 0.3882405745062837, 'f': 0.5559125964010283}}}



=== Document Evaluation Results ===

Prometheus Scores (0-1 scale, higher is better):

WriteHere Scores (0-1 scale, higher is better):

Article Quality Metrics:
Entity Recall (0-1 scale): N/A
Citation Count: 8

ROUGE Scores (precision, recall, F1):
rouge-1:
  p: 0.98
  r: 0.39
  f: 0.56
rouge-2:
  p: 0.96
  r: 0.38
  f: 0.54
rouge-l:
  p: 0.98
  r: 0.39
  f: 0.56


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Weak judge status: ok
Better judge status: ok


## 12. Inspect qualitative judge output

In [13]:
def print_judge(result, title):
    judge = result.get("dea_judge", {}) or {}
    print("="*90)
    print(title, "status=", judge.get("status"))
    if judge.get("error"):
        print("error:", judge.get("error"))
    print("\nAssessment:\n", judge.get("qualitative_assessment", ""))
    print("\nKeep:")
    for x in judge.get("keep", []):
        print("-", x.get("point"), "|", x.get("evidence"))
    print("\nProblems:")
    for x in judge.get("problems", []):
        print(f"- [{x.get('priority')}] {x.get('main_impact')}: {x.get('issue')}")
        print("  impact:", x.get("impact"))
        print("  confidence:", x.get("confidence"))
        print("  evidence:", x.get("evidence"))
    print("\nUncertainties:")
    for x in judge.get("uncertainties", []):
        print("-", x.get("question"), "=>", x.get("needed_evidence"))

print_judge(weak_with_judge, "Weak candidate")
print_judge(better_with_judge, "Better candidate")


Weak candidate status= ok

Assessment:
 The candidate document is severely underdeveloped relative to the gold standard. The main causes are an incomplete plan, extremely shallow content in key sections, and a missing bibliography/citations. This yields weak topical coverage, poor mapping to the gold plan, and minimal verifiability of sources.

Keep:
- Use of the central term 'Dual phase evolution' and alignment with the topic | "Dual phase evolution (DPE) is a process that promotes the emergence of large-scale order in complex systems."
- Presence of plan-like structure with labeled sections | h2 Introduction, h2 The DPE mechanism
- Inline citation marker after a claim | The following features are necessary for DPE to occur.[1]

Problems:
- [P0 blocker] plan: Plan incompleteness and misalignment with the gold plan
  impact: Without a complete, mapped plan, the document cannot demonstrate coverage of essential DPE topics (e.g., Underlying network, Phase shifts, Selection and variation,

## 13. Compact comparison table

In [14]:
import pandas as pd
rows=[]
for name, result in [("weak", weak_with_judge), ("better", better_with_judge)]:
    v = score_vector(result)
    j = result.get("dea_judge", {}) or {}
    rows.append({
        "candidate": name,
        **v,
        "judge_status": j.get("status"),
        "n_keep": len(j.get("keep", [])),
        "n_problems": len(j.get("problems", [])),
        "n_uncertainties": len(j.get("uncertainties", [])),
        "assessment_excerpt": (j.get("qualitative_assessment") or "")[:160],
    })
pd.DataFrame(rows)


,candidate,scalar,plan,content,references,rouge_l_f,citation_count,judge_status,n_keep,n_problems,n_uncertainties,assessment_excerpt
0,weak,0.004541,0.0,0.0,0.0,0.045415,1,ok,3,3,3,The candidate document is severely underdevelo...
1,better,0.055591,0.0,0.0,0.0,0.555913,8,ok,5,5,4,The candidate demonstrates a recognizable stru...


## 14. Convert scores + judge observations into an optimizer feedback packet

The judge does **not** prescribe fixes. It exposes observations that an optimizer can decide how to use.


In [15]:
def optimizer_feedback_packet(result):
    judge = result.get("dea_judge", {}) or {}
    return {
        "scores": score_vector(result),
        "qualitative_assessment": judge.get("qualitative_assessment", ""),
        "preserve": judge.get("keep", []),
        "problems_to_consider": judge.get("problems", []),
        "uncertainties": judge.get("uncertainties", []),
    }

pprint(optimizer_feedback_packet(weak_with_judge))


{'preserve': [{'evidence': '"Dual phase evolution (DPE) is a process that '
                           'promotes the emergence of large-scale order in '
                           'complex systems."',
               'point': "Use of the central term 'Dual phase evolution' and "
                        'alignment with the topic',
               'why_keep': 'Demonstrates basic topic recognition that could '
                           'guide future content expansion.'},
              {'evidence': 'h2 Introduction, h2 The DPE mechanism',
               'point': 'Presence of plan-like structure with labeled sections',
               'why_keep': 'Indicates awareness of conventional scholarly '
                           'structure that can support later development.'},
              {'evidence': 'The following features are necessary for DPE to '
                           'occur.[1]',
               'point': 'Inline citation marker after a claim',
               'why_keep': 'Shows an attempt

## 15. Optional direct `run_dea_judge(...)` usage

In [16]:
if SUPPORTS_DIRECT_DEA_JUDGE:
    direct_result = run_dea_judge(
        document_content=weak_candidate,
        solution=solution,
        content_type="markdown",
        dea_scores=weak_fast.get("dea_evaluation_scores", {}),
        article_metrics=weak_fast.get("article_metrics", {}),
        prometheus_scores=weak_fast.get("prometheus_scores", {}),
        writehere_scores=weak_fast.get("writehere_scores", {}),
        lm=fake_judge_lm if judge_mode == "fake_lm" else None,
        client=openrouter_client if judge_mode == "openrouter" else None,
        model=openrouter_model if judge_mode == "openrouter" else ("fake" if judge_mode == "fake_lm" else os.environ.get("OPENAI_JUDGE_MODEL", "gpt-5-nano")),
        max_prompt_chars=12000,
    )
    pprint(direct_result)
    if direct_result.get("status") != "ok":
        print("Direct judge returned non-ok status; this is safe and demonstrates error handling:", direct_result.get("error"))
else:
    print("Direct run_dea_judge is unavailable on this branch.")


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


{'keep': [{'evidence': '"Dual phase evolution (DPE) is a process that promotes '
                       'the emergence of large-scale order in complex '
                       'systems."',
           'point': 'Core DPE terminology carried in Introduction',
           'why_keep': 'Preserves the essential topic framing and terminology '
                       'that the gold solution centers on.'},
          {'evidence': '"The following features are necessary for DPE to '
                       'occur.[1]"',
           'point': 'Explicit citation tag in mechanism section',
           'why_keep': 'Maintains alignment with the gold plan’s citation '
                       'style and signaling of sources.'},
          {'evidence': 'Green, D.G.; Liu, J. & Abbass, H. (2014). Dual Phase '
                       'Evolution: from Theory to Practice. Berlin: Springer. '
                       'ISBN 978-1-4419-8422-7.',
           'point': 'Bibliography contains a major source from the gold set',
 

## 16. Optional low-level DEA JSON comparison helpers

In [17]:
if RUN_LOW_LEVEL_COMPARISON:
    try:
        from common.test_utils import compare_documents_sections, compare_documents_content
        print("Comparing target JSON to itself...")
        pprint(compare_documents_sections(solution, solution))
        pprint(compare_documents_content(solution, solution))
    except Exception as exc:
        print("Low-level comparison skipped/failed:", exc)
else:
    print("RUN_LOW_LEVEL_COMPARISON is False; skipping low-level comparison helpers.")


RUN_LOW_LEVEL_COMPARISON is False; skipping low-level comparison helpers.


## 17. End of demo

Suggested next experiments:

1. Replace `weak_candidate` and `better_candidate` with your own generated Markdown or LaTeX.
2. Set `RUN_FULL_DEA=True` for semantic target-distance metrics.
3. Compare OpenRouter models by setting `OPENROUTER_MODEL`.
4. Feed `optimizer_feedback_packet(...)` to your generative optimizer.
